# 🛒 Laboratorio 4 · Análisis Exploratorio de Datos (EDA)

**Universidad Técnica Nacional** · Minería de Datos ISW-911 · III Cuatrimestre 2026

| Integrantes |
|---|
| Jimena Jara |
| Walbyn González |
| Fiorella Lazo |
| Seidy Alanis |

---

### 🎯 Objetivo

Explorar la base de datos **original** de nuestro proyecto (*dunnhumby – The Complete Journey*): revisar su calidad,
encontrar datos faltantes y valores raros, tratarlos, y dejar un conjunto de datos limpio y listo para un modelo de
predicción.

### 📦 Los datos

Son las **8 tablas de origen** del proyecto (antes del ETL y del Data Warehouse). Están comprimidas en la carpeta
`datos/` del repositorio público: https://github.com/walbyn504/Laboratorio4-Mineria

| Tabla | ¿Qué guarda? | Ejemplo |
|---|---|---|
| `transaction_data` | Cada producto comprado en cada compra | "El hogar 2375 compró 1 papa a $1.39" |
| `product` | Información de cada producto | departamento, marca, categoría |
| `hh_demographic` | Perfil de los hogares que llenaron la encuesta | edad, ingreso, tamaño del hogar |
| `causal_data` | Si el producto estaba exhibido o en el folleto | "en la cabecera del pasillo, semana 10" |
| `campaign_table` | Qué hogares recibieron cada campaña de mercadeo | "el hogar 17 recibió la campaña 26" |
| `campaign_desc` | Cuándo empezó y terminó cada campaña | "campaña 24: del día 659 al 719" |
| `coupon` | Qué productos cubre cada cupón | |
| `coupon_redempt` | Qué cupones canjeó cada hogar y qué día | |

### 🧭 Cómo leer este cuaderno

Cada sección tiene tres partes:

1. **¿Qué hacemos?** — una explicación corta antes del código.
2. **Código** — con comentarios (`# ...`) que explican cada paso.
3. **📌 Qué encontramos** — las conclusiones, después de los resultados.

Ejecute las celdas **en orden**, de arriba hacia abajo (o use *Entorno de ejecución → Ejecutar todas*).

### 📖 Mini glosario

| Término | Significado sencillo |
|---|---|
| **Línea de venta** | Un producto dentro de una compra. Si alguien compra pan y leche, son 2 líneas. |
| **Nulo / NaN** | Una celda vacía: el dato no existe. |
| **Imputar** | Rellenar un dato vacío con un valor razonable. |
| **Valor atípico** | Un valor muy alejado de lo normal (por ejemplo, comprar 89 000 unidades). |
| **IQR** | Rango donde está el 50 % central de los datos. Se usa para detectar atípicos. |
| **Mediana** | El valor del medio cuando los datos se ordenan. No le afectan los valores extremos. |
| **Codificar** | Convertir texto en números para que un modelo pueda usarlo. |
| **Estandarizar** | Poner los números en una misma escala (media 0, desviación 1). |
| **Correlación** | Qué tanto se mueven juntas dos variables (de −1 a 1). |

---
## ⚙️ 0. Preparación

**¿Qué hacemos?** Cargamos las librerías y definimos un estilo para que todos los gráficos se vean iguales.

In [ ]:
import numpy as np                # cálculos numéricos
import pandas as pd               # manejo de tablas
import matplotlib.pyplot as plt   # gráficos
from matplotlib.ticker import FuncFormatter

# Mostrar todas las columnas de las tablas (sin cortarlas con "...")
pd.set_option("display.max_columns", None)

# ---- Estilo de los gráficos ----
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.spines.top": False,     # sin borde de arriba
    "axes.spines.right": False,   # sin borde de la derecha
})

# Colores que usamos en todo el cuaderno
AZUL, ROJO, VERDE, GRIS = "#4C72B0", "#DD5555", "#55A868", "#BBBBBB"

# Formato para mostrar los ejes en miles (1,000,000 -> 1,000)
en_miles = FuncFormatter(lambda x, _: f"{x / 1000:,.0f}")

# Dirección de la carpeta "datos" en nuestro GitHub
BASE = "https://raw.githubusercontent.com/walbyn504/Laboratorio4-Mineria/main/datos/"

---
## 📥 1. Carga de las tablas desde GitHub

**¿Qué hacemos?** Leemos las 8 tablas directamente desde GitHub. Los archivos están comprimidos (`.xz`) y pandas
los descomprime solo.

`causal_data` es la tabla más grande (casi **37 millones de filas**). Para que quepa en la memoria de Colab, le
indicamos tipos de datos "livianos": números pequeños para los códigos y `category` para los textos que se repiten.
Así ocupa unos 400 MB en lugar de varios GB.

In [ ]:
# 1. Las 7 tablas pequeñas y medianas
transacciones  = pd.read_csv(BASE + "transaction_data.csv.xz")
productos      = pd.read_csv(BASE + "product.csv.xz")
hogares        = pd.read_csv(BASE + "hh_demographic.csv.xz")
campanas_hogar = pd.read_csv(BASE + "campaign_table.csv.xz")
campanas       = pd.read_csv(BASE + "campaign_desc.csv.xz")
cupones        = pd.read_csv(BASE + "coupon.csv.xz")
canjes         = pd.read_csv(BASE + "coupon_redempt.csv.xz")

# 2. causal_data completa, con tipos livianos para ahorrar memoria
tipos_causal = {
    "PRODUCT_ID": "int32",    # código de producto
    "STORE_ID": "int32",      # código de tienda
    "WEEK_NO": "int8",        # número de semana (1 a 102)
    "display": "category",    # código de exhibición en tienda
    "mailer": "category",     # código de ubicación en el folleto
}
causal = pd.read_csv(BASE + "causal_data.csv.xz", dtype=tipos_causal)

print("✅ Las 8 tablas se cargaron correctamente")

In [ ]:
# Resumen de las 8 tablas: cuántas filas, columnas y celdas vacías tiene cada una
tablas = {
    "transaction_data": transacciones, "product": productos, "hh_demographic": hogares,
    "causal_data": causal, "campaign_table": campanas_hogar, "campaign_desc": campanas,
    "coupon": cupones, "coupon_redempt": canjes,
}

resumen_tablas = pd.DataFrame({
    "filas": {nombre: len(t) for nombre, t in tablas.items()},
    "columnas": {nombre: t.shape[1] for nombre, t in tablas.items()},
    "celdas_vacias": {nombre: int(t.isna().sum().sum()) for nombre, t in tablas.items()},
})
display(resumen_tablas)

In [ ]:
# Primeras 3 filas de cada tabla, para conocer su contenido
for nombre, tabla in tablas.items():
    print(f"\n📄 {nombre}")
    display(tabla.head(3))

### 🔗 Unir las 8 tablas en una sola

**¿Qué hacemos?** Juntamos todo en una tabla llamada `ventas`, donde **cada fila es una línea de venta**.
A cada venta le agregamos la información de las otras tablas:

| Tabla | ¿Cómo se conecta con la venta? | ¿Qué le agrega? |
|---|---|---|
| `product` | por el producto | departamento, marca, categoría, tamaño |
| `hh_demographic` | por el hogar | edad, estado civil, ingreso, vivienda, composición, tamaño del hogar, hijos |
| `causal_data` | por producto + tienda + semana | exhibición en tienda y ubicación en el folleto |
| `campaign_table` + `campaign_desc` | por el hogar y el día | cuántas campañas tenía activas el hogar ese día |
| `coupon` | por el producto | si el producto tiene algún cupón |
| `coupon_redempt` | por hogar + día + producto | si el hogar canjeó un cupón de ese producto ese día |

Usamos `left join`: **se conservan todas las ventas**, aunque no tengan información en la otra tabla
(en ese caso la celda queda vacía).

In [ ]:
llave_causal = ["PRODUCT_ID", "STORE_ID", "WEEK_NO"]

# 1. causal_data tiene algunas combinaciones repetidas: las quitamos para no duplicar ventas
repetidas = causal.duplicated(llave_causal).sum()
causal = causal.drop_duplicates(llave_causal)
print(f"Combinaciones repetidas eliminadas de causal_data: {repetidas:,}")

# 2. Unir ventas + productos + hogares + causal_data
columnas_producto = ["PRODUCT_ID", "DEPARTMENT", "BRAND", "COMMODITY_DESC", "CURR_SIZE_OF_PRODUCT"]

ventas = (
    transacciones
    .merge(productos[columnas_producto], on="PRODUCT_ID", how="left", validate="many_to_one")
    .merge(hogares, on="household_key", how="left", validate="many_to_one")
    .merge(causal, on=llave_causal, how="left", validate="many_to_one")
)
ventas["display"] = ventas["display"].astype("string")   # pasar a texto para tratarlas después
ventas["mailer"] = ventas["mailer"].astype("string")

In [ ]:
# 3. Campañas activas: para cada venta, contamos cuántas campañas tenía el hogar ese día.
#    Una campaña está activa si el hogar la recibió y el día de compra está entre su inicio y su fin.
periodos = campanas_hogar.merge(campanas[["CAMPAIGN", "START_DAY", "END_DAY"]], on="CAMPAIGN")

campanas_activas = np.zeros(len(ventas), dtype="int8")
for _, campana in periodos.groupby("CAMPAIGN"):            # recorrer las 30 campañas
    inicio, fin = campana["START_DAY"].iloc[0], campana["END_DAY"].iloc[0]
    hogar_la_recibio = ventas["household_key"].isin(campana["household_key"])
    dia_dentro = ventas["DAY"].between(inicio, fin)
    campanas_activas += (hogar_la_recibio & dia_dentro).to_numpy(dtype="int8")
ventas["ACTIVE_CAMPAIGNS"] = campanas_activas

# 4. ¿El producto tiene algún cupón? (1 = sí, 0 = no)
ventas["PRODUCT_HAS_COUPON"] = ventas["PRODUCT_ID"].isin(cupones["PRODUCT_ID"]).astype("int8")

# 5. ¿El hogar canjeó un cupón de ese producto ese día? (1 = sí, 0 = no)
canjes_por_producto = (
    canjes.merge(cupones, on=["COUPON_UPC", "CAMPAIGN"])     # cupón -> productos que cubre
    [["household_key", "DAY", "PRODUCT_ID"]]
    .drop_duplicates()
    .assign(COUPON_REDEEMED=1)
)
ventas = ventas.merge(canjes_por_producto, on=["household_key", "DAY", "PRODUCT_ID"], how="left")
ventas["COUPON_REDEEMED"] = ventas["COUPON_REDEEMED"].fillna(0).astype("int8")

# 6. Comprobación: la tabla final debe tener exactamente las mismas filas que transaction_data
assert len(ventas) == len(transacciones), "¡Las uniones cambiaron la cantidad de ventas!"
del causal   # liberar memoria

print(f"✅ Tabla de ventas lista: {ventas.shape[0]:,} filas y {ventas.shape[1]} columnas")
print(f"   Ventas con dato en causal_data: {ventas['display'].notna().sum():,}")
display(ventas.head())

---
## 🔍 2. Características principales

**¿Qué hacemos?** Usamos las funciones básicas de pandas para conocer la tabla:
`info()` (tipos y memoria), `nunique()` (valores distintos), `describe()` (estadísticas),
`head()`/`tail()`/`sample()` (ver filas), `duplicated()` (repetidos) y `value_counts()` (frecuencias).

In [ ]:
ventas.info()

In [ ]:
# Una fila por columna: su tipo, cuántos valores distintos tiene y un ejemplo
resumen_columnas = pd.DataFrame({
    "tipo": ventas.dtypes.astype(str),
    "valores_distintos": ventas.nunique(),
    "ejemplo": ventas.iloc[0],
})
display(resumen_columnas)

In [ ]:
# Últimas filas y 5 filas al azar
display(ventas.tail(3))
display(ventas.sample(5, random_state=42))

In [ ]:
# Estadísticas de las columnas numéricas (promedio, mínimo, máximo, cuartiles...)
display(ventas.describe().round(2))

In [ ]:
# Estadísticas de las columnas de texto (valores distintos, el más frecuente y cuántas veces aparece)
display(ventas.describe(exclude="number"))

In [ ]:
print("Filas duplicadas:       ", f"{ventas.duplicated().sum():,}")
print("Hogares distintos:      ", f"{ventas['household_key'].nunique():,}")
print("Compras distintas:      ", f"{ventas['BASKET_ID'].nunique():,}")
print("Productos distintos:    ", f"{ventas['PRODUCT_ID'].nunique():,}")
print("Días cubiertos:         ", ventas["DAY"].min(), "a", ventas["DAY"].max())

In [ ]:
def barras(serie, eje, titulo, color=AZUL):
    """Dibuja un gráfico de barras con el valor (en miles) encima de cada barra."""
    serie.plot.bar(ax=eje, rot=0, color=color, width=0.7)
    eje.set_title(titulo)
    eje.set_xlabel("")
    eje.set_ylabel("miles de líneas")
    eje.yaxis.set_major_formatter(en_miles)
    eje.bar_label(eje.containers[0], labels=[f"{v / 1000:,.0f} mil" for v in serie],
                  fontsize=8, padding=3)
    eje.margins(y=0.15)


fig, ejes = plt.subplots(2, 2, figsize=(15, 9))

# Departamentos con más ventas (barras horizontales para leer bien los nombres)
ventas["DEPARTMENT"].value_counts().head(10).sort_values().plot.barh(ax=ejes[0, 0], color=AZUL)
ejes[0, 0].set_title("Top 10 departamentos")
ejes[0, 0].set_xlabel("miles de líneas")
ejes[0, 0].set_ylabel("")
ejes[0, 0].xaxis.set_major_formatter(en_miles)

barras(ventas["BRAND"].value_counts(), ejes[0, 1], "Tipo de marca")

orden_edad = ["19-24", "25-34", "35-44", "45-54", "55-64", "65+"]
barras(ventas["AGE_DESC"].value_counts().reindex(orden_edad), ejes[1, 0],
       "Edad del jefe de hogar (solo hogares con perfil)")

barras(ventas["ACTIVE_CAMPAIGNS"].value_counts().sort_index(), ejes[1, 1],
       "Campañas activas del hogar el día de la compra")

fig.suptitle("¿Cómo se reparten las ventas?", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

### 📌 Qué encontramos

* La tabla tiene **2 595 732 líneas de venta** de **2 500 hogares**, en **276 484 compras** durante **711 días**.
  No hay filas duplicadas.
* **GROCERY** (abarrotes) concentra la mayoría de las ventas, y casi 3 de cada 4 productos vendidos son de **marca nacional**.
* `causal_data` tiene 36.8 millones de registros, pero solo una parte pequeña coincide con alguna venta: el resto son
  productos exhibidos o anunciados en semanas en que nadie del estudio los compró. Tenía **15 245 combinaciones repetidas**,
  que eliminamos.
* El **70 %** de las ventas ocurre cuando el hogar no tiene ninguna campaña activa.
* Las columnas de hogares y de `causal_data` tienen muchas celdas vacías: lo vemos en la sección 4.

---
## 🇪🇸 3. Nombres de columnas en español

**¿Qué hacemos?** Cambiamos el nombre de cada columna con un diccionario `{"nombre_original": "nombre_nuevo"}`.

In [ ]:
nombres_en_espanol = {
    # transaction_data
    "household_key": "id_hogar",          "BASKET_ID": "id_compra",
    "DAY": "dia",                         "PRODUCT_ID": "id_producto",
    "QUANTITY": "cantidad",               "SALES_VALUE": "importe_venta",
    "STORE_ID": "id_tienda",              "RETAIL_DISC": "descuento_tienda",
    "TRANS_TIME": "hora_transaccion",     "WEEK_NO": "semana",
    "COUPON_DISC": "descuento_cupon",     "COUPON_MATCH_DISC": "igualacion_descuento_cupon",
    # product
    "DEPARTMENT": "departamento",         "BRAND": "tipo_marca",
    "COMMODITY_DESC": "categoria",        "CURR_SIZE_OF_PRODUCT": "tamano_producto",
    # hh_demographic
    "AGE_DESC": "edad",                   "MARITAL_STATUS_CODE": "estado_civil",
    "INCOME_DESC": "ingreso",             "HOMEOWNER_DESC": "tipo_vivienda",
    "HH_COMP_DESC": "composicion_hogar",  "HOUSEHOLD_SIZE_DESC": "tamano_hogar",
    "KID_CATEGORY_DESC": "cantidad_hijos",
    # causal_data
    "display": "exhibicion",              "mailer": "folleto",
    # campañas y cupones
    "ACTIVE_CAMPAIGNS": "campanas_activas",
    "PRODUCT_HAS_COUPON": "producto_con_cupon",
    "COUPON_REDEEMED": "cupon_canjeado",
}

ventas = ventas.rename(columns=nombres_en_espanol)
print(list(ventas.columns))

---
## 🕳️ 4. Valores nulos

**¿Qué hacemos?** Buscamos las celdas vacías. En esta base aparecen de dos formas:

* **`NaN`**: celdas vacías de verdad. Aparecen al unir tablas, por ejemplo, cuando un hogar no tiene perfil demográfico.
* **Textos con un espacio** (`" "`): en `product`, algunos productos tienen el departamento o el tamaño escrito como
  un espacio. Pandas lo toma como texto válido, así que primero lo convertimos en `NaN`.

In [ ]:
def es_blanco(serie):
    """Devuelve True en las celdas que solo tienen espacios."""
    return serie.astype("string").str.strip().eq("").fillna(False).astype(bool)


# 1. Contar los textos en blanco de cada columna de texto
columnas_texto = ventas.select_dtypes(exclude="number").columns
blancos = {c: int(es_blanco(ventas[c]).sum()) for c in columnas_texto}
print("Textos en blanco encontrados:", {c: n for c, n in blancos.items() if n > 0})

# 2. Convertirlos en NaN
for columna in columnas_texto:
    ventas[columna] = ventas[columna].mask(es_blanco(ventas[columna]))

In [ ]:
# Tabla de nulos: cuántos tiene cada columna y qué porcentaje representan
nulos = pd.DataFrame({
    "nulos": ventas.isna().sum(),
    "porcentaje": (100 * ventas.isna().mean()).round(2),
}).sort_values("porcentaje", ascending=False)

display(nulos[nulos["nulos"] > 0])
print(f"Columnas sin ningún nulo: {(nulos['nulos'] == 0).sum()} de {len(nulos)}")

In [ ]:
# Gráfico: porcentaje de nulos por columna (rojo = tiene nulos, gris = completa)
porcentajes = nulos["porcentaje"].sort_values()

ax = porcentajes.plot.barh(figsize=(10, 9), color=[ROJO if p > 0 else GRIS for p in porcentajes])
ax.set_title("Porcentaje de valores nulos por columna")
ax.set_xlabel("% de líneas de venta")
ax.set_xlim(0, 100)
for barra, valor in zip(ax.containers[0], porcentajes):
    if valor > 0:
        ax.text(valor + 1, barra.get_y() + barra.get_height() / 2, f"{valor:.1f} %", va="center")
plt.tight_layout()
plt.show()

**¿Faltan juntos?** El siguiente gráfico agrupa las ventas según **qué grupo de datos les falta**.
Por ejemplo, "Falta: datos causales" son las ventas que tienen todo excepto exhibición y folleto.

In [ ]:
# Grupos de columnas que vienen de la misma tabla
grupos = {
    "perfil del hogar": ["edad", "estado_civil", "ingreso", "tipo_vivienda",
                         "composicion_hogar", "tamano_hogar", "cantidad_hijos"],
    "datos causales": ["exhibicion", "folleto"],
    "tamaño del producto": ["tamano_producto"],
    "clasificación del producto": ["departamento", "categoria"],
}

# Para cada venta: ¿le falta algún dato de cada grupo? (True/False)
falta = pd.DataFrame({grupo: ventas[cols].isna().any(axis=1) for grupo, cols in grupos.items()})


def describir(combinacion):
    """Convierte (True, False, ...) en un texto como 'Falta: perfil del hogar'."""
    faltan = [grupo for grupo, f in zip(falta.columns, combinacion) if f]
    return "Datos completos" if not faltan else "Falta: " + " + ".join(faltan)


combinaciones = falta.value_counts().head(8)                  # las 8 más frecuentes
combinaciones.index = [describir(c) for c in combinaciones.index]
combinaciones = combinaciones.sort_values()

ax = combinaciones.plot.barh(
    figsize=(12, 5), color=[VERDE if i == "Datos completos" else ROJO for i in combinaciones.index])
ax.set_title("¿Qué datos le faltan a cada venta?")
ax.set_xlabel("miles de líneas")
ax.xaxis.set_major_formatter(en_miles)
ax.bar_label(ax.containers[0], padding=4,
             labels=[f"{v:,.0f}  ({100 * v / len(ventas):.1f} %)" for v in combinaciones])
ax.margins(x=0.4)
plt.tight_layout()
plt.show()

# Comprobar si las columnas de cada grupo faltan siempre en las mismas filas
for grupo in ["perfil del hogar", "datos causales"]:
    vacias = ventas[grupos[grupo]].isna()
    print(f"¿Las columnas de '{grupo}' faltan siempre juntas?",
          "Sí" if vacias.all(axis=1).equals(vacias.any(axis=1)) else "No")

### 📌 Qué encontramos

| Grupo | Ventas sin dato | ¿Por qué? |
|---|---|---|
| **Datos causales** (exhibición y folleto) | 2 031 958 (**78 %**) | No hay registro en `causal_data` para ese producto, tienda y semana |
| **Perfil del hogar** (7 columnas) | 1 168 429 (**45 %**) | Solo 801 de los 2 500 hogares llenaron la encuesta |
| **Tamaño del producto** | 466 746 (**18 %**) | Venía en blanco en el catálogo de productos |
| **Departamento y categoría** | 7 839 (**0.3 %**) | Productos sin clasificar en el catálogo |

* Las columnas de cada grupo faltan **siempre juntas**: no son errores sueltos, sino información que nunca existió.
* Solo el **9.6 %** de las ventas tiene todos los datos completos.
* Las columnas de `transaction_data` (cantidad, importe, descuentos) **no tienen nulos**.
* El tratamiento se hace en la sección 6.

---
## 📈 5. Valores atípicos

**¿Qué hacemos?** Buscamos valores muy alejados de lo normal con la regla del **IQR**, la misma que dibujan los
diagramas de caja:

> Un valor es atípico si es **menor que Q1 − 1.5 × IQR** o **mayor que Q3 + 1.5 × IQR**,
> donde Q1 es el valor del 25 %, Q3 el del 75 % e IQR = Q3 − Q1.

También revisamos registros que no tienen lógica, como una venta de 0 unidades.

In [ ]:
columnas_numericas = ["cantidad", "importe_venta", "descuento_tienda",
                      "descuento_cupon", "igualacion_descuento_cupon"]


def resumen_atipicos_iqr(tabla, columnas):
    """Calcula los límites IQR de cada columna y cuántos valores quedan fuera."""
    filas = []
    for columna in columnas:
        q1, q3 = tabla[columna].quantile([0.25, 0.75])
        iqr = q3 - q1
        inferior, superior = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        atipicos = ((tabla[columna] < inferior) | (tabla[columna] > superior)).sum()
        filas.append({"columna": columna, "Q1": q1, "Q3": q3, "IQR": iqr,
                      "límite_inferior": inferior, "límite_superior": superior,
                      "atípicos": atipicos, "%": round(100 * atipicos / len(tabla), 2)})
    return pd.DataFrame(filas).set_index("columna")


display(resumen_atipicos_iqr(ventas, columnas_numericas).round(2))

In [ ]:
# Diagramas de caja: la caja es el 50 % central y los puntos rojos son los atípicos.
# Se usa escala "symlog" porque hay valores desde negativos hasta 89 000.
fig, ejes = plt.subplots(1, len(columnas_numericas), figsize=(16, 5))

for columna, eje in zip(columnas_numericas, ejes):
    valores = ventas[columna]
    q1, q3 = valores.quantile([0.25, 0.75])
    iqr = q3 - q1
    atipicos = valores[(valores < q1 - 1.5 * iqr) | (valores > q3 + 1.5 * iqr)].unique()

    eje.boxplot(valores, showfliers=False, widths=0.5)
    eje.scatter(np.ones(len(atipicos)), atipicos, s=8, alpha=0.35, color=ROJO)
    eje.set_yscale("symlog", linthresh=1)
    eje.set_title(columna, fontsize=10)
    eje.set_xticks([])

fig.suptitle("Diagramas de caja · atípicos en rojo", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Las 10 ventas con mayor cantidad y las 10 con mayor importe
columnas_ver = ["id_compra", "departamento", "categoria", "cantidad", "importe_venta"]
print("🔝 Mayor cantidad")
display(ventas.nlargest(10, "cantidad")[columnas_ver])
print("🔝 Mayor importe")
display(ventas.nlargest(10, "importe_venta")[columnas_ver])

In [ ]:
# ¿En qué departamentos están las cantidades enormes?
cantidad_por_depto = (ventas.groupby("departamento")["cantidad"]
                      .agg(mediana="median", maximo="max")
                      .sort_values("maximo", ascending=False).head(8))
display(cantidad_por_depto)

ax = cantidad_por_depto["maximo"].sort_values().plot.barh(figsize=(9, 4), logx=True, color=AZUL)
ax.set_title("Cantidad máxima vendida por departamento (escala logarítmica)")
ax.set_xlabel("cantidad máxima")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
# Registros que revisamos con lógica del negocio
revision = pd.Series({
    "Cantidad 0 (error)": (ventas["cantidad"] == 0).sum(),
    "Importe 0 (descuento total, válido)": (ventas["importe_venta"] == 0).sum(),
})

ax = revision.plot.barh(figsize=(9, 2.8), color=[ROJO, VERDE])
ax.set_title("Registros revisados por lógica del negocio")
ax.set_xlabel("líneas de venta")
ax.bar_label(ax.containers[0], fmt="{:,.0f}", padding=4)
ax.margins(x=0.2)
plt.tight_layout()
plt.show()

### 📌 Qué encontramos

* **Cantidad:** casi todas las ventas son de 1 unidad, así que Q1 = Q3 = 1 y el **IQR es 0**. Con eso, cualquier
  venta de 2 unidades ya sale como "atípica", lo cual no tiene sentido. Por eso, para tratarla usamos **percentiles**.
* Las cantidades enormes (hasta **89 638**) vienen de **KIOSK-GAS** (combustible) y **MISC SALES TRAN**, donde la
  cantidad no se mide en unidades como en el resto de productos.
* **Descuentos:** la mayoría vale 0, así que el IQR marca como atípico cualquier descuento. Son valores reales: se conservan.
* **14 466 ventas tienen cantidad 0**: es un error (no se puede vender 0 unidades). Las trataremos como dato faltante.
* **18 850 ventas tienen importe 0**: son productos con descuento total. Son válidas: se conservan.

---
## 🩹 6. Funciones de imputación

**¿Qué hacemos?** Rellenamos o corregimos los problemas encontrados en las secciones 4 y 5.
Esta tabla resume cada decisión y **por qué** la tomamos:

| Problema | Qué hacemos | ¿Por qué? |
|---|---|---|
| Perfil del hogar vacío (45 %) | Rellenar con **"Desconocido"** y crear la columna `tiene_perfil` (1/0) | Falta casi la mitad. Si rellenáramos con el valor más común, diríamos que casi la mitad de los hogares tienen la misma edad o ingreso, y eso es inventar. |
| Exhibición y folleto vacíos (78 %) | Rellenar con **"Sin dato"** y crear la columna `tiene_dato_causal` (1/0) | No sabemos si el producto estaba exhibido. Poner "No exhibido" sería afirmar algo que no sabemos. |
| Departamento y categoría vacíos (0.3 %) | Rellenar con **"SIN CLASIFICAR"** | Son pocos, pero ponerles "GROCERY" (el más común) los mezclaría con abarrotes. |
| Tamaño del producto vacío (18 %) | **Eliminar la columna** | Es texto libre con más de 4 000 formatos ("16 OZ", "1 LB", "12 CT"...). No se puede comparar ni rellenar bien. |
| Cantidad 0 (14 466 ventas) | Rellenar con la **mediana de su categoría** | Cada tipo de producto se compra en cantidades distintas. La mediana no se ve afectada por los valores enormes. |
| Cantidades enormes | **Recortar** al percentil 1 y 99 | Limita el efecto del combustible sin borrar ventas. |
| Importes y descuentos altos | **Se conservan** | Son reales. El importe es lo que queremos predecir y no debe modificarse. |

In [ ]:
def imputar_categorica(tabla, columna, estrategia="constante", valor="Desconocido"):
    """Rellena las celdas vacías de una columna de texto.

    - estrategia="constante": usa el valor que le indiquemos (ej. "Desconocido").
    - estrategia="moda": usa el valor que más se repite.
    Devuelve cuántas celdas rellenó.
    """
    vacias = tabla[columna].isna()
    relleno = tabla[columna].mode()[0] if estrategia == "moda" else valor
    tabla[columna] = tabla[columna].fillna(relleno)
    return int(vacias.sum())


def imputar_mediana_por_grupo(tabla, columna, grupo):
    """Rellena las celdas vacías de una columna numérica con la mediana de su grupo.

    Ejemplo: una venta de 'SOFT DRINKS' sin cantidad recibe la cantidad típica de 'SOFT DRINKS'.
    Si un grupo entero está vacío, usa la mediana general.
    """
    vacias = tabla[columna].isna()
    medianas = tabla.groupby(grupo, observed=True)[columna].transform("median")
    tabla[columna] = tabla[columna].fillna(medianas).fillna(tabla[columna].median())
    return int(vacias.sum())


def recortar_atipicos(tabla, columna, inferior=0.01, superior=0.99):
    """Recorta los valores extremos: lo que pasa del percentil 99 baja a ese valor
    y lo que está debajo del percentil 1 sube a ese valor. No borra filas."""
    limite_inf, limite_sup = tabla[columna].quantile([inferior, superior])
    fuera = ((tabla[columna] < limite_inf) | (tabla[columna] > limite_sup)).sum()
    tabla[columna] = tabla[columna].clip(limite_inf, limite_sup)
    return int(fuera), limite_inf, limite_sup

In [ ]:
nulos_antes = ventas.isna().sum()
columnas_demograficas = grupos["perfil del hogar"]

# 1. Perfil del hogar -> indicador + "Desconocido"
ventas["tiene_perfil"] = ventas["edad"].notna().astype("int8")
for columna in columnas_demograficas:
    n = imputar_categorica(ventas, columna, valor="Desconocido")
print(f"✔ Perfil del hogar: {n:,} ventas rellenadas con 'Desconocido' (7 columnas)")

# 2. Datos causales -> indicador + "Sin dato"
ventas["tiene_dato_causal"] = ventas["exhibicion"].notna().astype("int8")
for columna in ["exhibicion", "folleto"]:
    n = imputar_categorica(ventas, columna, valor="Sin dato")
print(f"✔ Exhibición y folleto: {n:,} ventas rellenadas con 'Sin dato'")

# 3. Productos sin clasificar -> "SIN CLASIFICAR"
for columna in ["departamento", "categoria"]:
    n = imputar_categorica(ventas, columna, valor="SIN CLASIFICAR")
print(f"✔ Departamento y categoría: {n:,} ventas rellenadas con 'SIN CLASIFICAR'")

# 4. Tamaño del producto -> se elimina la columna
ventas = ventas.drop(columns="tamano_producto")
print("✔ tamano_producto: columna eliminada")

# 5. Cantidad 0 -> se marca como vacía -> mediana de su categoría
ventas["cantidad"] = ventas["cantidad"].mask(ventas["cantidad"] == 0)
nulos_antes["cantidad"] = ventas["cantidad"].isna().sum()
n = imputar_mediana_por_grupo(ventas, "cantidad", "categoria")
print(f"✔ Cantidad 0: {n:,} ventas corregidas con la mediana de su categoría")

# 6. Cantidades enormes -> recorte entre percentil 1 y 99
n, li, ls = recortar_atipicos(ventas, "cantidad")
print(f"✔ Cantidades extremas: {n:,} ventas recortadas al rango [{li:.0f}, {ls:.0f}]")

print(f"\n🎉 Nulos restantes en toda la tabla: {ventas.isna().sum().sum()}")

In [ ]:
# Gráfico: celdas vacías antes y después de imputar
antes_despues = pd.DataFrame({
    "Antes": nulos_antes.drop("tamano_producto"),
    "Después": ventas.isna().sum().reindex(nulos_antes.index).drop("tamano_producto"),
})
antes_despues = antes_despues[antes_despues["Antes"] > 0].sort_values("Antes")

ax = antes_despues.plot.barh(figsize=(10, 6), color=[ROJO, VERDE], width=0.8)
ax.set_title("Celdas vacías antes y después de imputar\n(después de imputar quedan 0 en todas)")
ax.set_xlabel("miles de líneas")
ax.xaxis.set_major_formatter(en_miles)
plt.tight_layout()
plt.show()

---
## 🔤 7. Conversión de tipos de datos

**¿Qué hacemos?** Le damos a cada columna el tipo de dato que le corresponde:

* **Códigos de exhibición y folleto** (`"0"`, `"A"`, `"D"`...): los traducimos a su significado, según la guía de dunnhumby.
* **Estado civil** (`A`, `B`, `U`): según la misma guía, **A = Casado**, **B = Soltero** y **U = Desconocido**.
* **Hora** (`1631` = 16:31): extraemos solo la hora (16).
* **Identificadores** (hogar, compra, producto, tienda): son códigos, no cantidades. Los pasamos a texto para que no
  se sumen ni se promedien por error.
* **Columnas de texto con pocos valores**: las pasamos a `category`, que ocupa mucha menos memoria.
* **Cantidad**: vuelve a número entero después de la imputación.
* La base original **no tiene fechas**, solo el número de `dia` (1 a 711) y de `semana`: se quedan como enteros.

In [ ]:
memoria_antes = ventas.memory_usage(deep=True).sum() / 1024**2

# 1. Traducir los códigos de causal_data
codigos_exhibicion = {
    "0": "No exhibido",             "1": "Frente de tienda",
    "2": "Fondo de tienda",          "3": "Cabecera frontal",
    "4": "Cabecera pasillo central", "5": "Cabecera trasera",
    "6": "Cabecera pasillo lateral", "7": "En el pasillo",
    "9": "Ubicación secundaria",     "A": "En el estante",
    "Sin dato": "Sin dato",
}
codigos_folleto = {
    "0": "No anunciado",                "A": "Destacado página interior",
    "C": "Línea en página interior",    "D": "Destacado portada",
    "F": "Destacado contraportada",     "H": "Destacado envoltura frontal",
    "J": "Cupón envoltura interior",    "L": "Destacado envoltura trasera",
    "P": "Cupón página interior",       "X": "Gratis en página interior",
    "Z": "Gratis en portada",           "Sin dato": "Sin dato",
}
ventas["exhibicion"] = ventas["exhibicion"].astype(str).map(codigos_exhibicion)
ventas["folleto"] = ventas["folleto"].astype(str).map(codigos_folleto)

# Estado civil: según la guía de dunnhumby, A = casado, B = soltero, U = desconocido
codigos_estado_civil = {"A": "Casado", "B": "Soltero", "U": "Desconocido", "Desconocido": "Desconocido"}
ventas["estado_civil"] = ventas["estado_civil"].astype(str).map(codigos_estado_civil)

# 2. Hora: 1631 -> 16
ventas["hora"] = (ventas["hora_transaccion"] // 100).astype("int8")

# 3. Identificadores -> texto
for columna in ["id_hogar", "id_compra", "id_producto", "id_tienda"]:
    ventas[columna] = ventas[columna].astype("string")

# 4. Texto con pocos valores -> category
columnas_categoricas = (["departamento", "tipo_marca", "categoria", "exhibicion", "folleto"]
                        + columnas_demograficas)
for columna in columnas_categoricas:
    ventas[columna] = ventas[columna].astype("category")

# 5. Números enteros más livianos
ventas["cantidad"] = ventas["cantidad"].round().astype("int64")
ventas["dia"] = ventas["dia"].astype("int16")
ventas["semana"] = ventas["semana"].astype("int8")

memoria_despues = ventas.memory_usage(deep=True).sum() / 1024**2
print(f"💾 Memoria: {memoria_antes:,.0f} MB → {memoria_despues:,.0f} MB")
display(ventas.dtypes.to_frame("tipo"))

In [ ]:
# Ahora que los códigos tienen nombre, vemos cómo se distribuyen (sin contar "Sin dato")
titulos = {"exhibicion": "Exhibición en la tienda", "folleto": "Ubicación en el folleto"}

fig, ejes = plt.subplots(1, 2, figsize=(15, 5))
for columna, eje in zip(titulos, ejes):
    conteo = ventas.loc[ventas[columna] != "Sin dato", columna].value_counts()
    conteo[conteo > 0].sort_values().plot.barh(ax=eje, color=AZUL)
    eje.set_title(f"{titulos[columna]} (solo ventas con dato)")
    eje.set_xlabel("miles de líneas")
    eje.set_ylabel("")
    eje.xaxis.set_major_formatter(en_miles)
plt.tight_layout()
plt.show()

---
## 🔢 8. Variables categóricas a numéricas

**¿Qué hacemos?** Los modelos solo entienden números, así que convertimos el texto en números.
Usamos una técnica distinta según el tipo de variable:

| Técnica | ¿Cuándo se usa? | Ejemplo | Variables |
|---|---|---|---|
| **Binaria (0/1)** | La variable responde sí o no | Private → 1, National → 0 | `tipo_marca`, `exhibicion`, `folleto` |
| **Ordinal (1, 2, 3...)** | Las categorías tienen un orden | 19-24 → 1, 25-34 → 2 ... (Desconocido → 0) | `edad`, `ingreso`, `tamano_hogar`, `cantidad_hijos` |
| **One-hot** | Pocas categorías sin orden | Una columna 0/1 por cada categoría | `estado_civil`, `tipo_vivienda`, `composicion_hogar`, `exhibicion`, `folleto` |
| **Por etiqueta** | Demasiadas categorías | Cada categoría recibe un código | `departamento` (44), `categoria` (308) |

`departamento` y `categoria` no usan one-hot porque crearían más de 350 columnas para 2.6 millones de filas.

> 💡 **¿Y cómo leemos esos números?** Los 0 y 1 son para el **modelo**, no para las personas.
> Por eso, después de codificar, mostramos una tabla **"antes → después"** (el texto original al lado de su número)
> y un **diccionario** que dice qué significa cada número. Así no hay que adivinar.

In [ ]:
codificadas = pd.DataFrame(index=ventas.index)

# 1. Binarias: 1 = sí, 0 = no
codificadas["es_marca_propia"] = (ventas["tipo_marca"] == "Private").astype("int8")
codificadas["en_exhibicion"] = (~ventas["exhibicion"].isin(["No exhibido", "Sin dato"])).astype("int8")
codificadas["en_folleto"] = (~ventas["folleto"].isin(["No anunciado", "Sin dato"])).astype("int8")

# 2. Ordinales: se respeta el orden natural (Desconocido = 0)
ordenes = {
    "edad": ["19-24", "25-34", "35-44", "45-54", "55-64", "65+"],
    "ingreso": ["Under 15K", "15-24K", "25-34K", "35-49K", "50-74K", "75-99K",
                "100-124K", "125-149K", "150-174K", "175-199K", "200-249K", "250K+"],
    "tamano_hogar": ["1", "2", "3", "4", "5+"],
    "cantidad_hijos": ["None/Unknown", "1", "2", "3+"],
}
for columna, orden in ordenes.items():
    numeros = {valor: posicion for posicion, valor in enumerate(orden, start=1)}
    numeros["Desconocido"] = 0
    codificadas[f"{columna}_num"] = ventas[columna].astype(str).map(numeros).astype("int8")

# 3. Por etiqueta: cada categoría recibe un número
codificadas["departamento_cod"] = ventas["departamento"].cat.codes.astype("int16")
codificadas["categoria_cod"] = ventas["categoria"].cat.codes.astype("int16")

# 4. One-hot: una columna 0/1 por categoría
one_hot = pd.get_dummies(
    ventas[["estado_civil", "tipo_vivienda", "composicion_hogar", "exhibicion", "folleto"]],
    dtype="int8",
)
# Nombres de columna limpios: minúsculas, sin tildes ni espacios
one_hot.columns = (one_hot.columns.str.lower()
                   .str.normalize("NFKD").str.encode("ascii", "ignore").str.decode("ascii")
                   .str.replace(r"[^a-z0-9]+", "_", regex=True).str.strip("_"))

codificadas = pd.concat([codificadas, one_hot], axis=1)
print(f"✅ Se crearon {codificadas.shape[1]} columnas numéricas para el modelo")

### 👀 Antes → después: el texto original al lado de su número

Cada par de columnas muestra **lo que decía el dato** (texto) y **el número que recibe el modelo**.
Se toman 8 ventas al azar como ejemplo.

In [ ]:
pares = [("tipo_marca", "es_marca_propia"), ("exhibicion", "en_exhibicion"), ("folleto", "en_folleto"),
         ("edad", "edad_num"), ("ingreso", "ingreso_num"), ("tamano_hogar", "tamano_hogar_num"),
         ("estado_civil", "estado_civil_casado")]

muestra = ventas.sample(8, random_state=7).index
antes_despues = pd.DataFrame(index=muestra)
for texto, numero in pares:
    antes_despues[f"{texto} (texto)"] = ventas.loc[muestra, texto].astype(str)
    antes_despues[f"{numero} (número)"] = codificadas.loc[muestra, numero]

# Se colorean las columnas numéricas para distinguirlas del texto
columnas_numero = [c for c in antes_despues.columns if "(número)" in c]
display(antes_despues.reset_index(drop=True).style
        .set_properties(subset=columnas_numero, **{"background-color": "#e8f1fb", "font-weight": "bold"}))

### 📖 Diccionario: ¿qué significa cada número?

In [ ]:
diccionario = [
    {"columna": "es_marca_propia", "número": 1, "significa": "Marca propia de la tienda (Private)"},
    {"columna": "es_marca_propia", "número": 0, "significa": "Marca nacional (National)"},
    {"columna": "en_exhibicion", "número": 1, "significa": "El producto estaba exhibido en la tienda"},
    {"columna": "en_exhibicion", "número": 0, "significa": "No estaba exhibido o no hay dato"},
    {"columna": "en_folleto", "número": 1, "significa": "El producto salió en el folleto"},
    {"columna": "en_folleto", "número": 0, "significa": "No salió en el folleto o no hay dato"},
    {"columna": "estado_civil_casado", "número": 1, "significa": "Hogar casado (código A)"},
    {"columna": "estado_civil_soltero", "número": 1, "significa": "Hogar soltero (código B)"},
    {"columna": "columnas one-hot", "número": 1, "significa": "La venta SÍ pertenece a esa categoría"},
    {"columna": "columnas one-hot", "número": 0, "significa": "La venta NO pertenece a esa categoría"},
]
for columna, orden in ordenes.items():
    diccionario.append({"columna": f"{columna}_num", "número": 0, "significa": "Desconocido (hogar sin perfil)"})
    for posicion, valor in enumerate(orden, start=1):
        diccionario.append({"columna": f"{columna}_num", "número": posicion, "significa": valor})

diccionario = pd.DataFrame(diccionario)
pd.set_option("display.max_rows", 60)
display(diccionario.style.hide(axis="index"))

---
## 📏 9. Estandarización

**¿Qué hacemos?** Ponemos las variables numéricas en la misma escala con el **z-score**:

> **z = (valor − promedio) / desviación estándar**

Después de esto cada variable tiene promedio 0 y desviación 1. Así, una variable con valores grandes (como la cantidad)
no "pesa" más que otra con valores pequeños (como el descuento) solo por su escala.

* Elegimos **estandarizar** en vez de normalizar (min-max, de 0 a 1) porque hay valores extremos: con min-max casi
  todos los datos quedarían apretados cerca de 0.
* **No estandarizamos `importe_venta`**, porque es lo que queremos predecir y conviene dejarlo en dólares.

In [ ]:
def estandarizar_z(tabla, columnas):
    """Aplica z = (x - promedio) / desviación a cada columna. Agrega el sufijo _z."""
    datos = tabla[columnas].astype("float64")
    return ((datos - datos.mean()) / datos.std(ddof=0)).add_suffix("_z")


columnas_a_estandarizar = ["cantidad", "descuento_tienda", "descuento_cupon",
                           "igualacion_descuento_cupon", "campanas_activas"]
estandarizadas = estandarizar_z(ventas, columnas_a_estandarizar)

# Comprobación: el promedio debe ser 0 y la desviación 1
display(pd.DataFrame({"promedio": estandarizadas.mean(),
                      "desviación": estandarizadas.std(ddof=0)}).round(3))

In [ ]:
# Antes y después: la forma es la misma, solo cambia la escala del eje
fig, ejes = plt.subplots(1, 2, figsize=(12, 4))
ventas["descuento_tienda"].plot.hist(bins=60, ax=ejes[0], log=True, color=AZUL)
ejes[0].set_title("descuento_tienda (original, en $)")
estandarizadas["descuento_tienda_z"].plot.hist(bins=60, ax=ejes[1], log=True, color=VERDE)
ejes[1].set_title("descuento_tienda_z (estandarizado)")
for eje in ejes:
    eje.set_ylabel("líneas (escala log)")
plt.tight_layout()
plt.show()

In [ ]:
# Juntamos todo en la tabla final: datos originales + codificados + estandarizados
ventas_procesadas = pd.concat([ventas, codificadas, estandarizadas], axis=1)
del codificadas, estandarizadas
print(f"Tabla final: {ventas_procesadas.shape[0]:,} filas y {ventas_procesadas.shape[1]} columnas")

---
## 🔗 10. Correlación

**¿Qué hacemos?** Medimos qué tanto se relaciona cada variable con las demás usando la correlación de Pearson:

| Valor | Significado |
|---|---|
| cerca de **+1** | cuando una sube, la otra también sube |
| cerca de **0** | no hay relación lineal |
| cerca de **−1** | cuando una sube, la otra baja |

En el mapa de calor, **rojo** es relación positiva y **azul** es negativa.

In [ ]:
columnas_correlacion = [
    "importe_venta", "cantidad", "descuento_tienda", "descuento_cupon",
    "igualacion_descuento_cupon", "es_marca_propia", "en_exhibicion", "en_folleto",
    "tiene_dato_causal", "campanas_activas", "producto_con_cupon", "cupon_canjeado",
    "tiene_perfil", "edad_num", "ingreso_num", "tamano_hogar_num", "hora", "semana",
]
correlacion = ventas_procesadas[columnas_correlacion].corr()

fig, ax = plt.subplots(figsize=(13, 11))
imagen = ax.imshow(correlacion, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(columnas_correlacion)), columnas_correlacion, rotation=45, ha="right")
ax.set_yticks(range(len(columnas_correlacion)), columnas_correlacion)
ax.grid(False)
for i in range(len(columnas_correlacion)):
    for j in range(len(columnas_correlacion)):
        valor = correlacion.iloc[i, j]
        ax.text(j, i, f"{valor:.2f}", ha="center", va="center", fontsize=7,
                color="white" if abs(valor) > 0.65 else "black")
ax.set_title("Mapa de calor de correlaciones")
fig.colorbar(imagen, ax=ax, label="Correlación")
plt.tight_layout()
plt.show()

In [ ]:
# La columna más importante: ¿qué se relaciona con el importe de la venta?
con_importe = correlacion["importe_venta"].drop("importe_venta").sort_values()

ax = con_importe.plot.barh(figsize=(9, 6), color=np.where(con_importe < 0, ROJO, AZUL))
ax.set_title("Correlación de cada variable con importe_venta")
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("correlación")
plt.tight_layout()
plt.show()

### 📌 Qué encontramos

* **`cantidad`** es la que más se relaciona con el importe (**r ≈ 0.48**, positiva y moderada): más unidades, más dinero.
* **`descuento_tienda`** tiene **r ≈ −0.25**. Como los descuentos se guardan en negativo, significa que los productos
  con más descuento suelen ser los más caros.
* **Exhibición, folleto, campañas y cupones** casi no se relacionan con el importe (|r| < 0.05). Entre ellas sí:
  los productos en el folleto suelen tener descuento (r ≈ −0.33).
* **Las variables del hogar** están muy relacionadas entre sí (r de 0.65 a 0.88). Es en parte artificial: el 45 % de
  las ventas tiene "Desconocido" (0) en todas a la vez. Esto se llama **multicolinealidad** y afecta a los modelos
  lineales.
* Que la relación lineal sea débil no significa que no haya relación: puede ser no lineal o depender del tipo de producto.

---
## 🤖 11. Variable dependiente y modelo candidato

### 🎯 Variable dependiente: `importe_venta`

Es lo que queremos **predecir**: cuánto se cobra en cada línea de venta.

### ¿Clasificación o regresión?

| | Clasificación | Regresión |
|---|---|---|
| Predice... | una categoría (sí/no, tipo A/B/C) | un número |
| Nuestro caso | ❌ | ✅ `importe_venta` es un número continuo (dólares) |

### 🌳 Modelo propuesto: **Árbol de regresión**

* Las relaciones lineales son débiles (sección 10). Un árbol puede encontrar reglas como *"si la categoría es
  carne y está en el folleto, el importe suele ser X"*.
* Funciona bien con las variables codificadas por etiqueta (`categoria_cod`, `departamento_cod`).
* No necesita datos estandarizados y no le afecta la multicolinealidad de las variables del hogar.
* Es fácil de explicar: se pueden leer las reglas que llevan a cada predicción.

**Modelo de comparación:** una **regresión lineal múltiple**, para comprobar si el árbol realmente mejora el resultado.

⚠️ **Cuidado:** los descuentos y el canje de cupones se conocen en el momento de la venta. Si se quisiera predecir el
importe *antes* de la compra, habría que quitarlos del modelo.

---
## 💾 12. Guardar el conjunto de datos procesado

**¿Qué hacemos?** Guardamos la tabla final, con todo el EDA aplicado, en `data_process.csv`.

In [ ]:
import os

ventas_procesadas.to_csv("data_process.csv", index=False)

print("✅ Archivo guardado: data_process.csv")
print(f"   {ventas_procesadas.shape[0]:,} filas · {ventas_procesadas.shape[1]} columnas · "
      f"{os.path.getsize('data_process.csv') / 1024**2:,.0f} MB")

# Comprobación: volver a leer las primeras filas del archivo guardado
display(pd.read_csv("data_process.csv", nrows=3))

# Para descargarlo a la computadora (opcional, el archivo es grande):
# from google.colab import files
# files.download("data_process.csv")

---
## ✅ Resumen final

| Paso | Resultado |
|---|---|
| Carga | 8 tablas unidas en **2 595 732 ventas** |
| Nulos | Perfil del hogar (45 %), datos causales (78 %), tamaño (18 %), clasificación (0.3 %) |
| Atípicos | Cantidades de combustible (hasta 89 638) y 14 466 ventas con cantidad 0 |
| Imputación | Constantes + indicadores, mediana por categoría y recorte por percentiles → **0 nulos** |
| Codificación | Binaria, ordinal, one-hot y por etiqueta |
| Estandarización | z-score en 5 variables |
| Correlación | La cantidad (0.48) y el descuento de tienda (−0.25) son las más relacionadas con el importe |
| Modelo | **Regresión** con **árbol de regresión** para predecir `importe_venta` |
| Resultado | `data_process.csv` |